# Testing dan Prediction Request

**Nama:** sintiawati

**Username Dicoding:** sintiawati

Notebook ini digunakan untuk menguji endpoint model serving yang telah di-deploy di cloud dan melakukan prediction request.

## 1. Import Library

In [ ]:
import json
import base64
import requests
import tensorflow as tf

# Ganti dengan URL Railway app Anda
BASE_URL = "https://chicago-taxi-mlops-production.up.railway.app"

print(f"Base URL: {BASE_URL}")

## 2. Health Check

Mengecek status TF Serving via endpoint metadata model.
TF Serving tidak memiliki endpoint `/` seperti Flask.

In [ ]:
response = requests.get(f"{BASE_URL}/v1/models/taxi-model")
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## 3. Model Info

In [ ]:
response = requests.get(f"{BASE_URL}/v1/models/taxi-model")
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## 4. Prediction Request

Mengirim sample data untuk mendapatkan prediksi apakah penumpang akan memberikan tip.

In [ ]:
# Helper: convert dict fitur ke tf.Example serialized base64
FEATURE_NAMES = ["trip_miles", "fare", "trip_seconds", "trip_start_timestamp",
                "payment_type", "company", "trip_start_hour", "trip_start_day",
                "trip_start_month"]

def make_tf_example(row):
    feature = {
        "trip_miles": tf.train.Feature(float_list=tf.train.FloatList(value=[row["trip_miles"]])),
        "fare": tf.train.Feature(float_list=tf.train.FloatList(value=[row["fare"]])),
        "trip_seconds": tf.train.Feature(int64_list=tf.train.Int64List(value=[row["trip_seconds"]])),
        "trip_start_timestamp": tf.train.Feature(int64_list=tf.train.Int64List(value=[row["trip_start_timestamp"]])),
        "payment_type": tf.train.Feature(bytes_list=tf.train.BytesList(value=[row["payment_type"].encode()])),
        "company": tf.train.Feature(bytes_list=tf.train.BytesList(value=[row["company"].encode()])),
        "trip_start_hour": tf.train.Feature(int64_list=tf.train.Int64List(value=[row["trip_start_hour"]])),
        "trip_start_day": tf.train.Feature(int64_list=tf.train.Int64List(value=[row["trip_start_day"]])),
        "trip_start_month": tf.train.Feature(int64_list=tf.train.Int64List(value=[row["trip_start_month"]])),
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return base64.b64encode(example.SerializeToString()).decode("utf-8")

# Sample 1: Credit Card (expected: high tip probability)
sample_cc = {
    "trip_miles": 2.5,
    "fare": 8.5,
    "trip_seconds": 600,
    "trip_start_timestamp": 1437076800,
    "payment_type": "Credit Card",
    "company": "Taxi Affiliation Services",
    "trip_start_hour": 14,
    "trip_start_day": 3,
    "trip_start_month": 5,
}

# Sample 2: Cash (expected: low tip probability)
sample_cash = {
    "trip_miles": 1.2,
    "fare": 5.85,
    "trip_seconds": 180,
    "trip_start_timestamp": 1382319000,
    "payment_type": "Cash",
    "company": "Taxi Affiliation Services",
    "trip_start_hour": 1,
    "trip_start_day": 2,
    "trip_start_month": 10,
}

payload = {"instances": [
    {"examples": {"b64": make_tf_example(sample_cc)}},
    {"examples": {"b64": make_tf_example(sample_cash)}},
]}

print(json.dumps(payload, indent=2)[:200] + "...")

response = requests.post(
    f"{BASE_URL}/v1/models/taxi-model:predict",
    json=payload
)

print(f"\nStatus: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    print(f"Result: {json.dumps(result, indent=2)}")
    for i, pred in enumerate(result["predictions"]):
        prob = pred[0]
        print(f"Sample {i+1}: probability={prob:.4f}, prediction={'Tip' if prob > 0.5 else 'No Tip'}")
else:
    print(f"Error: {response.text}")

## 5. Batch Prediction Test

In [ ]:
# Batch test: load 5 samples dari CSV, filter ke 9 fitur model, kirim tf.Example
import pandas as pd

df = pd.read_csv("data/data.csv")
test_samples = df[FEATURE_NAMES].head(5).to_dict(orient="records")

# Konversi kolom int64 yang mungkin terbaca sebagai float di CSV
for s in test_samples:
    for col in ["trip_seconds", "trip_start_timestamp", "trip_start_hour", "trip_start_day", "trip_start_month"]:
        s[col] = int(s[col])

instances = [{"examples": {"b64": make_tf_example(s)}} for s in test_samples]
payload = {"instances": instances}

response = requests.post(
    f"{BASE_URL}/v1/models/taxi-model:predict",
    json=payload
)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    print(f"Predictions: {json.dumps(result, indent=2)}")
    for i, pred in enumerate(result["predictions"]):
        prob = pred[0]
        print(f"Sample {i+1}: probability={prob:.4f}, prediction={'Tip' if prob > 0.5 else 'No Tip'}")
else:
    print(f"Error: {response.text}")

## 6. Kesimpulan

Notebook ini berhasil menguji endpoint model serving yang telah di-deploy di cloud. Seluruh prediction request berhasil diproses oleh model.